# Configuration handling

## Parameters vs. hyperparameters

Machine learning models have two kinds of parameters:

- Model parameters, that are modified automatically during training
- [Hyperparameters](https://en.wikipedia.org/wiki/Hyperparameter_(machine_learning)), that describe the training procedure

Dealing with model parameters is easy as they are the weights that the model learns during the training procedure, but dealing with hyperparameters can be a lot trickier, because there is no automatic way of specifying them.

Hyperparameters include things like:

- Model structure parameters (model architecture, number of layers, shape of layers)
- Dataset parameters (datasets used for training)
- Pre-processing parameters (augmentation, normalization, pre-processing procedures)
- Optimizer parameters (optimizer type, optimizer learning rate, loss function specification)
- Training parameters (random seed initialization, number of training epochs, checkpoint intervals, training strategy)
- Tracking parameters (tracked quantities, tracking rate)

So whenever you're testing out different settings for your model you're doing [hyperparamter optimization](https://en.wikipedia.org/wiki/Hyperparameter_optimization) or [ablation studies](https://en.wikipedia.org/wiki/Ablation_(artificial_intelligence)).

This all can be summarized by a single meme:

[<img src="always_has_been.jpg" width="800"/>](always_has_been.jpg)

## Hyperparameters and reproducibility

Nowadays when you're publishing a model you're not only publishing the model parameters: you're also releasing the hyperparameters used to specify the model.

Just having the model parameters does not help other do inference or further training on your model. They need to be able to tell what is the model structure and what other hyperparameters they should set.

It can be that the hyperparameters are [hard coded](https://en.wikipedia.org/wiki/Hard_coding) to your codebase or it can be that they are specified in a configuration file. Either way, you'll have to release hyperparameters to make the model usable.

For the purpose of reproducibility it is highly preferable to separate the hyperparameters from the code.

Advantages of this approach include:
- Hyperparameters can be modified without doing codebase changes
- You do not need to go through the codebase to find out key model parameters
- Different experiments can be easily compared by comparing the configuration files
- Hyperparameters are stored in a format that is easy to take under version control

## Managing hyperparameters via configuration: OmegaConf

In this lesson we'll teach how to use [OmegaConf](https://omegaconf.readthedocs.io/), which is a library that can easily handle YAML based hierarchical configurations. This is widely used by many machine learning codebases.

### Creating a configuration from a dictonary

Let's create a sample configuration from a simple dictionary and let's consider this to be our default configuration values:

In [ ]:
from omegaconf import OmegaConf

default_conf = OmegaConf.create({
    "dataset": {
        "datamodule": "MNISTDataModule",
        "batch_size": 32
    },
    "optimizer": {
        "type": "AdamW", "lr": 0.001
    },
    "outputs": {
        "output_dir": "outputs",
    }
})

print(OmegaConf.to_yaml(default_conf))

A common convention is to use [snake case](https://en.wikipedia.org/wiki/Snake_case) for variable names (lower case characters separated by underscore). This is same as [Python's recommended variable naming style](https://peps.python.org/pep-0008/#function-and-variable-names).

### Reading configurations from a file

In most cases you'll want to specify the configuration as a YAML file from the start and read it in. Let's read an example configuration.

In [ ]:
with open("config.yaml", "r") as f:
    conf = OmegaConf.load(f)

print(OmegaConf.to_yaml(conf))

### Reading configurations from the command line

You can also read configurations from the command line:

In [ ]:
# Fake system arguments
import sys
sys.argv = ['training_script.py', 'dataset.batch_size=64', 'experiment_name=exp1']

cli_conf = OmegaConf.from_cli()
print(OmegaConf.to_yaml(cli_conf))

### Merging configurations

A key feature of OmegaConf is that you can merge configurations from multiple sources:

In [ ]:
merged_conf = OmegaConf.merge(default_conf, conf, OmegaConf.from_cli())
print(OmegaConf.to_yaml(merged_conf))

As you can read from the configuration file, it is quite easy to see what sort of model we're dealing with:
- We're loading data from a MNIST datamodule with a batch size of 64.
- Our optimizer is AdamW with learning rate of 0.001.
- We're training for 10 epochs.
- Our model is a simple multilayer perceptron with a single layer with a hidden size of 20.
- Our outputs go to a directory called `outputs`.
- Our experiment is called `exp1`.

### Saving configurations

We can now save this configuration as an experiment:

In [ ]:
with open("exp1.yaml", "w") as f:
    OmegaConf.save(merged_conf, f)

### Using configuration in our code

You can access the values of your configuration with multiple syntaxes, but attribute-style is the most commonly used:

In [ ]:
# Accessing with attribute-style access
print(merged_conf.dataset.batch_size)

# Accessing with dictionary-style access
print(merged_conf['dataset']['batch_size'])

# Accessing nested configuration values
dataset_conf = merged_conf.dataset
print(dataset_conf.batch_size)

### Additional features of OmegaConf

OmegaConf supports features such as:
- Structured configurations with type checking
- Configuration interpolation

## Experiment handling with Hydra

[Hydra](https://hydra.cc/) is a library created by Facebook for handling machine learning experiments.

It uses OmegaConf under the hood for configuration management, but it provides additional features that make experiment handling easier.

Lets create an example application that shows our configuration file. It is shown in `hydra_example.py`:

```python
import hydra
from omegaconf import OmegaConf, DictConfig


@hydra.main(version_base="1.2", config_path="configs", config_name="config")
def app(cfg):
    print(OmegaConf.to_yaml(cfg))


if __name__ == "__main__":
    app()
```

Here `hydra.main`-decorator creates the configuration handling automatically for our application.

Arguments for it contains:

- `version_base` specifies which version of Hydra we're using.
- `config_path` specifies the path where our configs reside.
- `config_name` specifies the name of the configuration file.

Lets run the application:

In [ ]:
%run hydra_example.py

We now see that Hydra read the `config.yaml`-file from our configuration directory.

It also created a new directory called `outputs` and inside that directory it created a new folder structure where the names of the folders are based on the timestamp when we ran the command.

When we run a Hydra app, Hydra will:

- Create an output folder for our application (in code this is `hydra.core.hydra_config.HydraConfig.get().runtime.output_dir`)
- It will copy the used configuration to a folder called `.hydra` inside the output directory
- It will set up a log file automatically for our application

Lets add logging into our example:
```python
import hydra
import logging
from omegaconf import OmegaConf

# Se up logging for this file
log = logging.getLogger(__name__)


@hydra.main(version_base="1.2", config_path="configs", config_name="config")
def app(cfg):

    log.info("Starting the app")
    print(OmegaConf.to_yaml(cfg))


if __name__ == "__main__":
    app()
```

In [ ]:
%run hydra_example2.py

Hydra supports command line overrides in [a similar fashion as OmegaConf](https://hydra.cc/docs/tutorials/basic/your_first_app/config_file/).

### Initializing Hydra in Jupyter notebooks

Hydra can be used in Jupyter notebooks using a [Compose API](https://hydra.cc/docs/advanced/compose_api/).

When used in this manner some of the advanced features like output folder management and multiple runners cannot be used.

Here's an example of the previous code:

In [ ]:
from hydra import initialize, compose
import logging
from omegaconf import OmegaConf

# Se up logging for this file
log = logging.getLogger(__name__)

def app(cfg):

    log.info("Starting the app")
    print(OmegaConf.to_yaml(cfg))


if __name__ == "__main__":
    with initialize(version_base="1.2", config_path="configs"):
        cfg = compose(config_name="config")
        app(cfg)

### Additional features

Hydra supports additional features such as:
- Structured configurations with type checking
- Hierarchical configurations
- Run sweeps across multiple parameters
- Custom configuration all Hydra defaults such as output folder formats, log file formats etc.
- [Parallel launcher of job tasks](https://hydra.cc/docs/plugins/joblib_launcher/) via [Joblib](https://joblib.readthedocs.io/en/stable/)
- [Slurm launcher plugin](https://hydra.cc/docs/plugins/submitit_launcher/) via [submitit](https://github.com/facebookincubator/submitit)

## Handling secrets with dotenv

In some cases your code might need to use secrets like passwords, server names etc. that you do not want to store in the configuration files.

A commonly used solution to this is to use a so-called dotenv-file and the [dotenv-package](https://github.com/theskumar/python-dotenv).

What this package does it that the it reads a file called `.env` from your directory and it will add those variables into your program's environment variables. There they can be accessed 

It is **highly** recommended to add the `.env`-file into your `.gitignore` so that your secrets are not added into your code repository. You should also make certain that your AI agents cannot access this file.

Lets create a sample `.env`-file:

In [ ]:
with open(".env", "w") as f:
    f.write("SECRET_KEY=supersecret")

Let's read it with dotenv:

In [ ]:
import os
import dotenv


dotenv.load_dotenv()
print(os.environ['SECRET_KEY'])

### Combining dotenv with OmegaConf

OmegaConf can interpolate configuration values from various sources. One of these sources is to use [environment variables](https://omegaconf.readthedocs.io/en/2.3_branch/custom_resolvers.html#oc-env).

Thus we can combine dotenv and OmegaConf easily.

Let's create a simple OmegaConf configuration. Do note that value of the interpolated environment value is not set to the configuration itself, but it is resolved once it is retrieved:

In [ ]:
interpolated_conf = OmegaConf.create({
    "secret_key": "${oc.env:SECRET_KEY}"
})

# The configuration template does not contain the actual value of the environment variable
print(OmegaConf.to_yaml(interpolated_conf))

In [ ]:
# Accessing the interpolated value of the environment variable
print(interpolated_conf.secret_key)